# Module 05 — Neural Network Foundations
## 05-03: Activation Functions

**Objective:** Implement ReLU, Sigmoid, Tanh, LeakyReLU, GELU, and Swish from scratch; understand their properties, the vanishing gradient problem, and how activation choice affects MLP training on FashionMNIST.

**Prerequisites:** 05-01 (Perceptron), 05-02 (Multilayer Perceptrons)

## Part 0 — Setup & Prerequisites

This notebook covers the **activation function landscape** — the non-linear transformations that enable neural networks to learn complex, non-linear mappings. Without activations, composing any number of linear layers reduces to a single linear transformation.

We implement six activation functions and their gradients from scratch in NumPy, verify them against `torch.nn.functional`, demonstrate the vanishing gradient problem for deep networks, then train FashionMNIST MLPs with each activation and compare training dynamics and final accuracy.

**Prerequisites:** 05-01 (Perceptron), 05-02 (Multilayer Perceptrons)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from typing import Callable

import random
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility & Device ─────────────────────────────────────────────────

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
BATCH_SIZE        = 256
NUM_EPOCHS        = 8
LEARNING_RATE     = 1e-3
DATA_DIR          = "../../data"
INPUT_DIM         = 784          # FashionMNIST: 28×28 flattened
NUM_CLASSES       = 10
HIDDEN_DIM        = 256
LEAKY_ALPHA       = 0.01         # Negative slope for LeakyReLU

FASHION_CLASSES = [
    "T-shirt/top", "Trouser",  "Pullover", "Dress",      "Coat",
    "Sandal",      "Shirt",    "Sneaker",  "Bag",         "Ankle boot",
]

ACTIVATION_NAMES = ["ReLU", "Sigmoid", "Tanh", "LeakyReLU", "GELU", "Swish"]
ACT_COLORS       = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

print(f"Activations: {', '.join(ACTIVATION_NAMES)}")
print(f"FashionMNIST: {INPUT_DIM}-D input → {NUM_CLASSES} classes  |  hidden={HIDDEN_DIM}")

### Data Loading & EDA

Two datasets are used throughout this notebook:
- **`make_moons`** — toy 2-D binary classification for decision-boundary visualisation.
- **FashionMNIST** — 70K greyscale 28×28 images across 10 apparel categories.

In [ ]:
# ── make_moons (2-D toy demo) ────────────────────────────────────────────────
X_moons, y_moons = make_moons(n_samples=2000, noise=0.2, random_state=SEED)
scaler_moons     = StandardScaler()
X_moons          = scaler_moons.fit_transform(X_moons).astype(np.float32)
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_moons, y_moons, test_size=0.2, random_state=SEED, stratify=y_moons
)
print(f"make_moons — train: {len(X_tr_m)}  test: {len(X_te_m)}")

# ── FashionMNIST ──────────────────────────────────────────────────────────────
_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.2860,), std=(0.3530,)),
])
full_train_set = datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=True, transform=_transform
)
test_set_fm = datasets.FashionMNIST(
    root=DATA_DIR, train=False, download=True, transform=_transform
)
generator    = torch.Generator().manual_seed(SEED)
train_size   = int(0.9 * len(full_train_set))
val_size     = len(full_train_set) - train_size
train_set_fm, val_set_fm = torch.utils.data.random_split(
    full_train_set, [train_size, val_size], generator=generator
)
train_loader = DataLoader(train_set_fm, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_set_fm,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_set_fm,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
print(f"FashionMNIST — train: {train_size:,}  val: {val_size:,}  test: {len(test_set_fm):,}")

# ── EDA: sample grid ──────────────────────────────────────────────────────────
fig_eda, axes_eda = plt.subplots(2, 8, figsize=(14, 4))
for ax_e, (img_e, lbl_e) in zip(axes_eda.ravel(),
                                  [full_train_set[i] for i in range(16)]):
    ax_e.imshow(img_e.squeeze().numpy(), cmap="gray")
    ax_e.set_title(FASHION_CLASSES[lbl_e], fontsize=7)
    ax_e.axis("off")
fig_eda.suptitle("FashionMNIST Sample Images", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# ── EDA: class distribution ───────────────────────────────────────────────────
all_labels  = [full_train_set[i][1] for i in range(len(full_train_set))]
label_counts = np.bincount(all_labels)
fig_bar, ax_bar = plt.subplots(figsize=(10, 3))
ax_bar.bar(range(NUM_CLASSES), label_counts, color=plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES)))
ax_bar.set_xticks(range(NUM_CLASSES))
ax_bar.set_xticklabels(FASHION_CLASSES, rotation=35, ha="right", fontsize=9)
ax_bar.set_ylabel("Count")
ax_bar.set_title("FashionMNIST Class Distribution (Training Set)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()
print("Classes are balanced — 6,000 images per class.")

---
## Part 1 — Activation Functions from Scratch

### 1.1 Mathematical Landscape

An activation $f: \mathbb{R} \to \mathbb{R}$ is applied element-wise after each linear layer. Without it, stacking $L$ layers $\mathbf{h}_l = \mathbf{W}_l \mathbf{h}_{l-1} + \mathbf{b}_l$ collapses to a single affine map $\mathbf{h}_L = \mathbf{A}\mathbf{x} + \mathbf{c}$.

| Activation | Formula | Range | Smooth |
|------------|---------|-------|--------|
| ReLU | $\max(0, x)$ | $[0, +\infty)$ | No |
| Sigmoid | $\sigma(x) = \frac{1}{1+e^{-x}}$ | $(0, 1)$ | Yes |
| Tanh | $\tanh(x)$ | $(-1, 1)$ | Yes |
| LeakyReLU | $\max(\alpha x, x),\ \alpha{=}0.01$ | $\mathbb{R}$ | No |
| GELU | $x \cdot \Phi(x)$ | $\approx(-0.17, +\infty)$ | Yes |
| Swish | $x \cdot \sigma(x)$ | $\approx(-0.28, +\infty)$ | Yes |

### 1.2 The Vanishing Gradient Problem

During backpropagation, the gradient at layer $l$ is scaled by $f'(z_l)$. For Sigmoid:
$$\sigma'(x) = \sigma(x)(1 - \sigma(x)) \leq 0.25$$

Through 20 layers the gradient magnitude shrinks by up to $0.25^{20} \approx 10^{-12}$, effectively **freezing** early layers. ReLU avoids this with $f'(x) = 1$ for $x > 0$, though it introduces **dead neurons** ($f(x) = 0$ for $x \leq 0$).

In [ ]:
# ── NumPy activation implementations ─────────────────────────────────────────

def relu_np(x: np.ndarray) -> np.ndarray:
    '''Rectified Linear Unit: max(0, x).

    Args:
        x: Input array of any shape.

    Returns:
        Element-wise max(0, x); same shape as x.
    '''
    return np.maximum(0.0, x)


def relu_grad_np(x: np.ndarray) -> np.ndarray:
    '''Subgradient of ReLU: 1 where x > 0, else 0.

    Args:
        x: Input array.

    Returns:
        Array of 0s and 1s, same shape as x.
    '''
    return (x > 0).astype(np.float64)


def sigmoid_np(x: np.ndarray) -> np.ndarray:
    '''Logistic sigmoid: 1 / (1 + exp(-x)), numerically stable via clipping.

    Args:
        x: Input array.

    Returns:
        Values in (0, 1), same shape as x.
    '''
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500.0, 500.0)))


def sigmoid_grad_np(x: np.ndarray) -> np.ndarray:
    '''Gradient of sigmoid: sigma(x) * (1 - sigma(x)).

    Args:
        x: Input array.

    Returns:
        Derivative values in (0, 0.25], same shape as x.
    '''
    s = sigmoid_np(x)
    return s * (1.0 - s)


def tanh_np(x: np.ndarray) -> np.ndarray:
    '''Hyperbolic tangent.

    Args:
        x: Input array.

    Returns:
        Values in (-1, 1), same shape as x.
    '''
    return np.tanh(x)


def tanh_grad_np(x: np.ndarray) -> np.ndarray:
    '''Gradient of tanh: 1 - tanh^2(x).

    Args:
        x: Input array.

    Returns:
        Derivative values in (0, 1], same shape as x.
    '''
    return 1.0 - np.tanh(x) ** 2


def leaky_relu_np(x: np.ndarray, alpha: float = LEAKY_ALPHA) -> np.ndarray:
    '''Leaky ReLU: max(alpha*x, x).

    Args:
        x:     Input array.
        alpha: Negative-slope coefficient. Default LEAKY_ALPHA (0.01).

    Returns:
        Element-wise leaky rectification, same shape as x.
    '''
    return np.where(x >= 0.0, x, alpha * x)


def leaky_relu_grad_np(x: np.ndarray, alpha: float = LEAKY_ALPHA) -> np.ndarray:
    '''Gradient of Leaky ReLU: 1 where x >= 0, else alpha.

    Args:
        x:     Input array.
        alpha: Negative-slope coefficient. Default LEAKY_ALPHA (0.01).

    Returns:
        Array of 1s and alphas, same shape as x.
    '''
    return np.where(x >= 0.0, 1.0, alpha)


def gelu_np(x: np.ndarray) -> np.ndarray:
    '''Gaussian Error Linear Unit (tanh approximation).

    GELU(x) = x * 0.5 * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3))).
    This matches the approximation used in GPT and BERT.

    Args:
        x: Input array.

    Returns:
        GELU-activated values, same shape as x.
    '''
    c     = np.sqrt(2.0 / np.pi)
    inner = c * (x + 0.044715 * x ** 3)
    return x * 0.5 * (1.0 + np.tanh(inner))


def gelu_grad_np(x: np.ndarray) -> np.ndarray:
    '''Approximate derivative of GELU.

    Args:
        x: Input array.

    Returns:
        GELU derivative approximation, same shape as x.
    '''
    c       = np.sqrt(2.0 / np.pi)
    inner   = c * (x + 0.044715 * x ** 3)
    tanh_v  = np.tanh(inner)
    sech2   = 1.0 - tanh_v ** 2
    d_inner = c * (1.0 + 3.0 * 0.044715 * x ** 2)
    cdf     = 0.5 * (1.0 + tanh_v)
    pdf     = 0.5 * sech2 * d_inner
    return cdf + x * pdf


def swish_np(x: np.ndarray) -> np.ndarray:
    '''Swish (also called SiLU): x * sigmoid(x).

    Args:
        x: Input array.

    Returns:
        Swish-activated values, same shape as x.
    '''
    return x * sigmoid_np(x)


def swish_grad_np(x: np.ndarray) -> np.ndarray:
    '''Derivative of Swish: sigmoid(x) + x * sigmoid(x) * (1 - sigmoid(x)).

    Args:
        x: Input array.

    Returns:
        Swish derivative values, same shape as x.
    '''
    s = sigmoid_np(x)
    return s + x * s * (1.0 - s)


# ── Collect activation pairs ──────────────────────────────────────────────────
ACT_FUNCS: list[tuple[str, Callable, Callable, str]] = [
    ("ReLU",                    relu_np,          relu_grad_np,          "#1f77b4"),
    ("Sigmoid",                 sigmoid_np,       sigmoid_grad_np,       "#ff7f0e"),
    ("Tanh",                    tanh_np,          tanh_grad_np,          "#2ca02c"),
    (f"LeakyReLU(a={LEAKY_ALPHA})", leaky_relu_np, leaky_relu_grad_np,  "#d62728"),
    ("GELU",                    gelu_np,          gelu_grad_np,          "#9467bd"),
    ("Swish",                   swish_np,         swish_grad_np,         "#8c564b"),
]

# ── Properties table ──────────────────────────────────────────────────────────
x_probe = np.linspace(-5.0, 5.0, 2000)
print(f"{'Activation':<22s}  {'f(0)':>6s}  {'f(-2)':>7s}  {'f(2)':>6s}"
      f"  {'grad_max':>9s}  {'grad@0':>8s}  {'saturated%':>11s}")
print("-" * 82)
for name, fn, gfn, _ in ACT_FUNCS:
    f0        = fn(np.array([0.0]))[0]
    fm2       = fn(np.array([-2.0]))[0]
    fp2       = fn(np.array([2.0]))[0]
    g0        = gfn(np.array([0.0]))[0]
    grad_vals = gfn(x_probe)
    gmax      = grad_vals.max()
    # "saturated" = gradient < 10% of max
    sat_pct   = 100.0 * (grad_vals < 0.1 * gmax).mean()
    print(f"{name:<22s}  {f0:>6.3f}  {fm2:>7.3f}  {fp2:>6.3f}"
          f"  {gmax:>9.4f}  {g0:>8.4f}  {sat_pct:>10.1f}%")

In [ ]:
# ── Plot all 6 activations and their derivatives ─────────────────────────────
x_plot = np.linspace(-5.0, 5.0, 500)

fig, axes = plt.subplots(2, 6, figsize=(21, 6))
axes[0, 0].set_ylabel("f(x)", fontsize=11)
axes[1, 0].set_ylabel("f'(x)", fontsize=11)

for idx, (name, fn, gfn, color) in enumerate(ACT_FUNCS):
    y_val  = fn(x_plot)
    y_grad = gfn(x_plot)
    ax_top = axes[0, idx]
    ax_bot = axes[1, idx]

    ax_top.plot(x_plot, y_val,  color=color, lw=2.2)
    ax_top.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)
    ax_top.axvline(0, color="k", lw=0.6, ls="--", alpha=0.5)
    ax_top.set_title(name, fontsize=9, fontweight="bold")
    ax_top.set_xlim(-5, 5)
    ax_top.set_ylim(-2.3, 3.1)
    ax_top.tick_params(labelsize=7)

    ax_bot.plot(x_plot, y_grad, color=color, lw=2.2)
    ax_bot.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)
    ax_bot.axvline(0, color="k", lw=0.6, ls="--", alpha=0.5)
    ax_bot.set_xlim(-5, 5)
    ax_bot.set_ylim(-0.15, 1.55)
    ax_bot.tick_params(labelsize=7)

fig.suptitle("Activation Functions f(x) (top) and Derivatives f'(x) (bottom)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Overlay: all functions on same axes ──────────────────────────────────────
fig2, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(12, 4.5))
for name, fn, gfn, color in ACT_FUNCS:
    ax_l.plot(x_plot, fn(x_plot),  color=color, lw=1.8, label=name)
    ax_r.plot(x_plot, gfn(x_plot), color=color, lw=1.8, label=name)

for ax_ov in (ax_l, ax_r):
    ax_ov.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)
    ax_ov.axvline(0, color="k", lw=0.6, ls="--", alpha=0.5)
    ax_ov.legend(fontsize=8, loc="upper left")
    ax_ov.tick_params(labelsize=8)

ax_l.set_xlabel("x"); ax_l.set_ylabel("f(x)")
ax_l.set_title("Activation Functions — Overlay", fontsize=11, fontweight="bold")
ax_r.set_xlabel("x"); ax_r.set_ylabel("f'(x)")
ax_r.set_title("Derivatives — Overlay", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

### 1.3 Numerical Verification Against PyTorch

Our NumPy implementations should match `torch.nn.functional` to floating-point precision. We verify against `F.relu`, `torch.sigmoid`, `torch.tanh`, `F.leaky_relu`, `F.gelu`, and `F.silu` (SiLU ≡ Swish) over 200 evenly-spaced inputs.

In [ ]:
# ── Verify NumPy vs torch.nn.functional ──────────────────────────────────────
x_t = torch.linspace(-5.0, 5.0, 200)
x_n = x_t.numpy()

torch_fns: dict[str, np.ndarray] = {
    "ReLU":      F.relu(x_t).numpy(),
    "Sigmoid":   torch.sigmoid(x_t).numpy(),
    "Tanh":      torch.tanh(x_t).numpy(),
    "LeakyReLU": F.leaky_relu(x_t, negative_slope=LEAKY_ALPHA).numpy(),
    "GELU":      F.gelu(x_t).numpy(),
    "Swish":     F.silu(x_t).numpy(),   # SiLU ≡ Swish in PyTorch
}
np_fns: dict[str, np.ndarray] = {
    "ReLU":      relu_np(x_n),
    "Sigmoid":   sigmoid_np(x_n),
    "Tanh":      tanh_np(x_n),
    "LeakyReLU": leaky_relu_np(x_n),
    "GELU":      gelu_np(x_n),
    "Swish":     swish_np(x_n),
}

print("NumPy vs torch.nn.functional — max absolute error:")
print("-" * 45)
all_pass = True
for name in ACTIVATION_NAMES:
    max_err = float(np.abs(np_fns[name] - torch_fns[name]).max())
    ok      = max_err < 1e-5
    symbol  = "✓" if ok else "✗"
    print(f"  {symbol} {name:<12s}  max_err = {max_err:.3e}")
    if not ok:
        all_pass = False

print()
if all_pass:
    print("All six activations match torch within 1e-5. ✓")
else:
    print("One or more activations exceed tolerance — review implementation.")

### 1.4 Vanishing Gradient Simulation

We simulate a depth-20 forward-backward pass: at each layer we apply $\mathbf{z}_l = \mathbf{W}_l \mathbf{h}_{l-1}$, $\mathbf{h}_l = f(\mathbf{z}_l)$, then propagate gradients backward and record the gradient norm at each depth. We average over 50 random initialisations to smooth noise.

In [ ]:
# ── Vanishing gradient: simulate backward pass through depth-20 network ───────

def simulate_gradient_flow(
    activation_fn: Callable[[np.ndarray], np.ndarray],
    grad_fn:       Callable[[np.ndarray], np.ndarray],
    n_layers:      int = 20,
    n_neurons:     int = 100,
    n_trials:      int = 50,
    rng:           np.random.Generator | None = None,
) -> np.ndarray:
    '''Measure mean gradient norms through a deep linear+activation stack.

    Each layer: z = W @ h, h = activation(z).  Backward: grad propagated
    as grad = (grad * f'(z)) @ W.T.  Xavier initialisation (std=1/sqrt(n)).

    Args:
        activation_fn: NumPy forward activation.
        grad_fn:       NumPy gradient of activation.
        n_layers:      Depth of the simulated network.
        n_neurons:     Width of each layer.
        n_trials:      Independent initialisations to average over.
        rng:           Optional NumPy Generator for reproducibility.

    Returns:
        Array of shape (n_layers,) with mean gradient norm at each layer
        (index 0 = closest to output, index n_layers-1 = furthest from output).
    '''
    if rng is None:
        rng = np.random.default_rng(SEED)
    grad_norms = np.zeros((n_trials, n_layers))
    for trial in range(n_trials):
        weights   = [rng.normal(0.0, np.sqrt(1.0 / n_neurons), (n_neurons, n_neurons))
                     for _ in range(n_layers)]
        h = rng.standard_normal(n_neurons)
        pre_acts: list[np.ndarray] = []
        for w in weights:
            z = w @ h
            pre_acts.append(z)
            h = activation_fn(z)
        grad = np.ones(n_neurons)
        for layer_idx in range(n_layers - 1, -1, -1):
            da_dz   = grad_fn(pre_acts[layer_idx])
            grad    = (grad * da_dz) @ weights[layer_idx].T
            grad_norms[trial, layer_idx] = np.linalg.norm(grad)
    return grad_norms.mean(axis=0)


vg_acts: list[tuple[str, Callable, Callable, str]] = [
    ("Sigmoid",   sigmoid_np,    sigmoid_grad_np,    "#ff7f0e"),
    ("Tanh",      tanh_np,       tanh_grad_np,       "#2ca02c"),
    ("ReLU",      relu_np,       relu_grad_np,       "#1f77b4"),
    ("LeakyReLU", leaky_relu_np, leaky_relu_grad_np, "#d62728"),
    ("GELU",      gelu_np,       gelu_grad_np,       "#9467bd"),
    ("Swish",     swish_np,      swish_grad_np,      "#8c564b"),
]

print("Simulating gradient flow (50 trials × 20 layers × 100 neurons)…")
grad_profiles: dict[str, np.ndarray] = {}
for name, fn, gfn, _ in vg_acts:
    grad_profiles[name] = simulate_gradient_flow(
        fn, gfn, rng=np.random.default_rng(SEED + ord(name[0]))
    )
    print(f"  {name} done.")

# ── Plot ──────────────────────────────────────────────────────────────────────
depths = list(range(1, 21))
fig_vg, (ax_lin, ax_log) = plt.subplots(1, 2, figsize=(14, 5))
for (name, _, _, color), profile in zip(vg_acts, grad_profiles.values()):
    ax_lin.plot(depths, profile, marker="o", markersize=4, lw=2, color=color, label=name)
    ax_log.semilogy(depths, profile + 1e-30, marker="o", markersize=4, lw=2,
                    color=color, label=name)

for ax_vg, ylabel in [(ax_lin, "Mean Gradient Norm"), (ax_log, "Mean Gradient Norm (log)")]:
    ax_vg.set_xlabel("Layer depth from output", fontsize=11)
    ax_vg.set_ylabel(ylabel, fontsize=11)
    ax_vg.legend(fontsize=9)
    ax_vg.grid(True, alpha=0.3)
ax_lin.set_title("Linear scale", fontsize=11, fontweight="bold")
ax_log.set_title("Log scale", fontsize=11, fontweight="bold")
fig_vg.suptitle("Gradient Flow Through 20 Layers — Xavier Initialisation",
                fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nLayer-1 vs Layer-20 gradient norms:")
print(f"{'Activation':<12s}  {'Layer 1':>10s}  {'Layer 20':>12s}  {'Decay ratio':>13s}")
print("-" * 54)
for name, profile in grad_profiles.items():
    ratio = (profile[0] + 1e-30) / (profile[-1] + 1e-30)
    print(f"{name:<12s}  {profile[0]:>10.4f}  {profile[-1]:>12.2e}  {ratio:>13.1f}×")

---
## Part 2 — MLP with Configurable Activation

We build a clean `MLP` class that accepts any activation as an `nn.Module`, making it trivial to swap activations for comparison experiments.

In [ ]:
# ── FlattenView helper ────────────────────────────────────────────────────────
class FlattenView(nn.Module):
    '''Flatten spatial dims inside nn.Sequential, preserving batch size.'''

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Flatten all dims except the batch dim.

        Args:
            x: Input tensor of shape (N, ...).

        Returns:
            Tensor of shape (N, prod(remaining dims)).
        '''
        return x.view(x.size(0), -1)


# ── Configurable MLP ──────────────────────────────────────────────────────────
class MLP(nn.Module):
    '''Multi-layer perceptron with configurable depth, width, and activation.

    Attributes:
        net: The Sequential stack of layers.
    '''

    def __init__(
        self,
        input_dim:  int,
        hidden_dim: int,
        output_dim: int,
        n_hidden:   int,
        activation: nn.Module,
        dropout:    float = 0.0,
    ) -> None:
        '''Initialise MLP.

        Args:
            input_dim:  Number of input features.
            hidden_dim: Width of each hidden layer.
            output_dim: Number of output logits.
            n_hidden:   Number of hidden layers.
            activation: Activation nn.Module applied after each hidden linear.
            dropout:    Dropout probability (0 = no dropout).
        '''
        super().__init__()
        layers: list[nn.Module] = [FlattenView()]
        in_dim = input_dim
        for _ in range(n_hidden):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(activation)
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Compute logits for input x.

        Args:
            x: Input of shape (N, input_dim) or (N, C, H, W).

        Returns:
            Logits of shape (N, output_dim).
        '''
        return self.net(x)

    def count_parameters(self) -> int:
        '''Count the total number of trainable scalar parameters.

        Returns:
            Integer parameter count.
        '''
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def get_activation(name: str) -> nn.Module:
    '''Return a fresh nn.Module instance for the named activation function.

    Args:
        name: Activation name. One of ACTIVATION_NAMES.

    Returns:
        Corresponding nn.Module.

    Raises:
        ValueError: If name is not recognised.
    '''
    mapping: dict[str, nn.Module] = {
        "ReLU":      nn.ReLU(),
        "Sigmoid":   nn.Sigmoid(),
        "Tanh":      nn.Tanh(),
        "LeakyReLU": nn.LeakyReLU(negative_slope=LEAKY_ALPHA),
        "GELU":      nn.GELU(),
        "Swish":     nn.SiLU(),  # SiLU ≡ Swish in PyTorch
    }
    if name not in mapping:
        raise ValueError(f"Unknown activation '{name}'. Choose from {list(mapping)}")
    return mapping[name]


# ── Sanity check ──────────────────────────────────────────────────────────────
_test_model = MLP(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES, n_hidden=2,
                  activation=nn.ReLU()).to(device)
_dummy      = torch.randn(8, 1, 28, 28).to(device)
_out        = _test_model(_dummy)
assert _out.shape == (8, NUM_CLASSES), f"Expected (8, {NUM_CLASSES}), got {_out.shape}"
print(f"MLP sanity check passed.  Output: {_out.shape}")
print(f"Trainable parameters: {_test_model.count_parameters():,}")

### 2.1 Decision Boundaries on make_moons

We train a tiny 2-input, 2-hidden-layer MLP (width 16) on the moons dataset for 80 epochs with each activation and visualise the resulting decision boundaries. This reveals how the activation's non-linearity shapes the learned boundary.

In [ ]:
# ── TensorDataset for make_moons ─────────────────────────────────────────────
X_tr_t = torch.tensor(X_tr_m, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr_m, dtype=torch.long)
X_te_t = torch.tensor(X_te_m, dtype=torch.float32)
y_te_t = torch.tensor(y_te_m, dtype=torch.long)
moons_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=True)


def train_moons_model(
    activation_name: str,
    n_epochs:        int = 80,
) -> tuple[nn.Module, float]:
    '''Train a tiny MLP on make_moons and return the model and test accuracy.

    Args:
        activation_name: Activation function name (must be in ACTIVATION_NAMES).
        n_epochs:        Number of training epochs.

    Returns:
        Tuple of (trained_model, test_accuracy_float).
    '''
    act   = get_activation(activation_name)
    model = MLP(2, 16, 2, n_hidden=2, activation=act).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=5e-3)
    crit  = nn.CrossEntropyLoss()
    for _ in range(n_epochs):
        model.train()
        for xb, yb in moons_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        preds = model(X_te_t.to(device)).argmax(dim=1).cpu()
        acc   = (preds == y_te_t).float().mean().item()
    return model, acc


# ── Build mesh grid for boundary plots ───────────────────────────────────────
pad = 0.5
x_min, x_max = X_moons[:, 0].min() - pad, X_moons[:, 0].max() + pad
y_min, y_max = X_moons[:, 1].min() - pad, X_moons[:, 1].max() + pad
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 220),
                     np.linspace(y_min, y_max, 220))
grid_pts = torch.tensor(np.c_[xx.ravel(), yy.ravel()],
                        dtype=torch.float32).to(device)

fig_db, axes_db = plt.subplots(2, 3, figsize=(14, 9))
moons_rows: list[dict] = []

for idx, act_name in enumerate(ACTIVATION_NAMES):
    moons_model, moons_acc = train_moons_model(act_name)
    moons_rows.append({"Activation": act_name, "Test Acc": round(moons_acc, 4)})

    with torch.no_grad():
        probs = moons_model(grid_pts).softmax(dim=1)[:, 1].cpu().numpy()
    zz = probs.reshape(xx.shape)

    ax = axes_db.ravel()[idx]
    ax.contourf(xx, yy, zz, levels=50, cmap="RdBu_r", alpha=0.75)
    ax.contour(xx, yy, zz, levels=[0.5], colors="k", linewidths=1.2)
    for cls_idx, cls_color in enumerate(["#1f77b4", "#d62728"]):
        mask = y_te_m == cls_idx
        ax.scatter(X_te_m[mask, 0], X_te_m[mask, 1],
                   s=10, alpha=0.7, c=cls_color)
    ax.set_title(f"{act_name}  (acc={moons_acc:.3f})", fontsize=10, fontweight="bold")
    ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
    ax.tick_params(labelsize=7)

fig_db.suptitle("Decision Boundaries on make_moons — One Activation per Panel",
                fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

moons_df = pd.DataFrame(moons_rows).sort_values("Test Acc", ascending=False)
print("make_moons results (sorted by accuracy):")
print(moons_df.to_string(index=False))

---
## Part 3 — Training on FashionMNIST

We scale up to a two-hidden-layer MLP ($784 \to 256 \to 256 \to 10$) and train each activation variant for **8 epochs**, comparing validation loss and accuracy trajectories.

> Gradient descent basics: see `01-03` (Calculus). MLP architecture: see `05-02`.

In [ ]:
# ── Standard training functions (see RULES_CORE §10) ─────────────────────────

def train_one_epoch(
    model:      nn.Module,
    dataloader: DataLoader,
    optimizer:  torch.optim.Optimizer,
    criterion:  nn.Module,
    device:     torch.device,
) -> tuple[float, float]:
    '''Train model for one epoch.

    Args:
        model:      The neural network.
        dataloader: Training DataLoader.
        optimizer:  Optimiser instance.
        criterion:  Loss function.
        device:     Compute device.

    Returns:
        Tuple of (average_loss, accuracy) over the epoch.
    '''
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0
    for batch_inputs, batch_targets in dataloader:
        batch_inputs, batch_targets = (batch_inputs.to(device),
                                       batch_targets.to(device))
        optimizer.zero_grad()
        outputs = model(batch_inputs)
        loss    = criterion(outputs, batch_targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_inputs.size(0)
        _, predicted  = outputs.max(1)
        total        += batch_targets.size(0)
        correct      += predicted.eq(batch_targets).sum().item()
    return running_loss / total, correct / total


def evaluate(
    model:      nn.Module,
    dataloader: DataLoader,
    criterion:  nn.Module,
    device:     torch.device,
) -> tuple[float, float]:
    '''Evaluate model on a dataloader (no gradient computation).

    Args:
        model:      The neural network.
        dataloader: Validation or test DataLoader.
        criterion:  Loss function.
        device:     Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0
    with torch.no_grad():
        for batch_inputs, batch_targets in dataloader:
            batch_inputs, batch_targets = (batch_inputs.to(device),
                                           batch_targets.to(device))
            outputs      = model(batch_inputs)
            loss         = criterion(outputs, batch_targets)
            running_loss += loss.item() * batch_inputs.size(0)
            _, predicted  = outputs.max(1)
            total        += batch_targets.size(0)
            correct      += predicted.eq(batch_targets).sum().item()
    return running_loss / total, correct / total

In [ ]:
# ── Train one MLP per activation on FashionMNIST ─────────────────────────────
fm_results:   list[dict]            = []
fm_histories: dict[str, dict]       = {}

for act_name in ACTIVATION_NAMES:
    act_mod    = get_activation(act_name)
    model_fm   = MLP(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES,
                     n_hidden=2, activation=act_mod).to(device)
    opt_fm     = torch.optim.Adam(model_fm.parameters(), lr=LEARNING_RATE)
    crit_fm    = nn.CrossEntropyLoss()
    history: dict[str, list[float]] = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
    }
    best_val_loss    = float("inf")
    best_state       = None
    t_start_fm       = time.time()

    for epoch in range(NUM_EPOCHS):
        tr_loss, tr_acc = train_one_epoch(
            model_fm, train_loader, opt_fm, crit_fm, device
        )
        v_loss, v_acc = evaluate(model_fm, val_loader, crit_fm, device)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(v_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(v_acc)
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state    = {k: v.clone() for k, v in model_fm.state_dict().items()}
        if (epoch + 1) % 4 == 0 or epoch == 0:
            elapsed_fm = time.time() - t_start_fm
            print(f"[{act_name:<10s}] Epoch {epoch+1}/{NUM_EPOCHS} | "
                  f"Train Loss: {tr_loss:.4f} | Val Loss: {v_loss:.4f} | "
                  f"Val Acc: {v_acc:.2%} | Time: {elapsed_fm:.1f}s")

    model_fm.load_state_dict(best_state)
    test_loss_fm, test_acc_fm = evaluate(model_fm, test_loader, crit_fm, device)
    total_time_fm = time.time() - t_start_fm
    fm_results.append({
        "Activation":    act_name,
        "Val Acc":       round(history["val_acc"][-1], 4),
        "Test Acc":      round(test_acc_fm, 4),
        "Train Time(s)": round(total_time_fm, 1),
        "Params":        model_fm.count_parameters(),
    })
    fm_histories[act_name] = history
    print()

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig_tc, axes_tc = plt.subplots(1, 2, figsize=(13, 5))
epochs_range    = range(1, NUM_EPOCHS + 1)

for act_name, color in zip(ACTIVATION_NAMES, ACT_COLORS):
    hist = fm_histories[act_name]
    axes_tc[0].plot(epochs_range, hist["val_loss"], color=color, lw=2,
                    marker="o", markersize=5, label=act_name)
    axes_tc[1].plot(epochs_range, hist["val_acc"],  color=color, lw=2,
                    marker="o", markersize=5, label=act_name)

axes_tc[0].set_xlabel("Epoch"); axes_tc[0].set_ylabel("Validation Loss")
axes_tc[0].set_title("Validation Loss by Activation", fontsize=11, fontweight="bold")
axes_tc[0].legend(fontsize=9)
axes_tc[1].set_xlabel("Epoch"); axes_tc[1].set_ylabel("Validation Accuracy")
axes_tc[1].set_title("Validation Accuracy by Activation", fontsize=11, fontweight="bold")
axes_tc[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

fm_df = pd.DataFrame(fm_results).sort_values("Val Acc", ascending=False)
print("FashionMNIST results (sorted by Val Acc):")
print(fm_df.to_string(index=False))

### 3.2 Convergence Speed Comparison

How many epochs does each activation need to first exceed accuracy thresholds? We also decompose the total validation loss drop into first-half vs second-half contributions to see whether activations converge quickly early or late.


In [ ]:
# ── Convergence speed: epochs to exceed accuracy thresholds ──────────────────
thresholds = [0.70, 0.75, 0.80, 0.82, 0.84]

conv_rows: list[dict] = []
for act_name in ACTIVATION_NAMES:
    val_accs_h = fm_histories[act_name]["val_acc"]
    row: dict = {"Activation": act_name}
    for thresh in thresholds:
        first_epoch = next(
            (ep + 1 for ep, acc in enumerate(val_accs_h) if acc >= thresh),
            None,
        )
        row[f">={thresh:.0%}"] = first_epoch if first_epoch is not None else "—"
    row["Final Val"] = f"{val_accs_h[-1]:.4f}"
    row["Epochs<0.84"] = sum(1 for acc in val_accs_h if acc >= 0.84)
    conv_rows.append(row)

conv_df = pd.DataFrame(conv_rows)
print("Convergence speed (epoch index when val_acc first crosses threshold):")
print(conv_df.to_string(index=False))
print("'—' = threshold not reached within 8 epochs.")

# ── Visual: accuracy curves with threshold lines ───────────────────────────────
fig_cs, ax_cs = plt.subplots(figsize=(10, 5))
for act_name, color in zip(ACTIVATION_NAMES, ACT_COLORS):
    acc_curve = fm_histories[act_name]["val_acc"]
    ax_cs.plot(range(1, NUM_EPOCHS + 1), acc_curve,
               color=color, lw=2, marker="o", markersize=5, label=act_name)

ax_cs.axhline(0.80, color="gray",   lw=1.5, ls="--", alpha=0.8, label="80%")
ax_cs.axhline(0.82, color="silver", lw=1.5, ls=":",  alpha=0.8, label="82%")
ax_cs.set_xlabel("Epoch", fontsize=11)
ax_cs.set_ylabel("Validation Accuracy", fontsize=11)
ax_cs.set_title("Convergence Speed — Activation Comparison on FashionMNIST",
                fontsize=11, fontweight="bold")
ax_cs.legend(fontsize=9, ncol=2)
ax_cs.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax_cs.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

print("\nKey insight: modern smooth activations (GELU/Swish) often converge faster")
print("in the first few epochs because their smooth gradients give cleaner update directions.")
print("ReLU eventually closes the gap; Sigmoid may never reach the upper thresholds in few epochs.")

# ── Per-epoch loss reduction ──────────────────────────────────────────────────
loss_drops: list[dict] = []
for act_name in ACTIVATION_NAMES:
    hist = fm_histories[act_name]
    v_losses = hist["val_loss"]
    drop_ep1 = v_losses[0]                              # loss after epoch 1
    drop_final = v_losses[-1]                           # loss after last epoch
    drop_total = drop_ep1 - drop_final                  # total drop
    drop_first_half = v_losses[0] - v_losses[NUM_EPOCHS // 2 - 1]
    drop_second_half = v_losses[NUM_EPOCHS // 2 - 1] - v_losses[-1]
    loss_drops.append({
        "Activation":   act_name,
        "Loss@Ep1":     round(drop_ep1, 4),
        "Loss@Final":   round(drop_final, 4),
        "TotalDrop":    round(drop_total, 4),
        "1stHalf%":     round(drop_first_half / drop_total * 100 if drop_total > 0 else 0, 1),
        "2ndHalf%":     round(drop_second_half / drop_total * 100 if drop_total > 0 else 0, 1),
    })

loss_drop_df = pd.DataFrame(loss_drops).sort_values("TotalDrop", ascending=False)
print("\nValidation loss reduction breakdown:")
print(loss_drop_df.to_string(index=False))
print("1stHalf% = fraction of total improvement in first half of training.")


### Library Comparison

`get_activation()` delegates to PyTorch built-ins (`nn.ReLU`, `nn.GELU`, `nn.SiLU`, etc.). Section 1.3 verified that our NumPy implementations match these to < 1e-5 absolute error, so the training results above are implicitly validated against the library reference.

### 3.3 Activation Output Scale & Zero-Centring

Weight initialisation strategies (Xavier/He) assume specific relationships between input and output variance. Here we empirically measure how each activation's output statistics vary with input scale — motivating why **He initialisation** uses $\text{std} = \sqrt{2/n_{\text{in}}}$ for ReLU (the $\sqrt{2}$ compensates for the ~50% zero-out).


In [ ]:
# ── Activation output statistics vs input variance ───────────────────────────
# Motivation: weight initialisation strategies (He/Xavier) are designed to
# maintain activation variance close to 1 through depth.  Different activations
# have different output variance as a function of input variance.

input_stds    = np.array([0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0])
n_samples_sc  = 50_000
rng_sc        = np.random.default_rng(SEED + 42)

output_stats: dict[str, dict[str, list[float]]] = {
    name: {"mean": [], "std": [], "var": []}
    for name, *_ in ACT_FUNCS
}

for sigma in input_stds:
    x_in = rng_sc.normal(0.0, sigma, n_samples_sc)
    for name, fn, _, _ in ACT_FUNCS:
        y_out = fn(x_in)
        output_stats[name]["mean"].append(float(y_out.mean()))
        output_stats[name]["std"].append(float(y_out.std()))
        output_stats[name]["var"].append(float(y_out.var()))

# ── Plot: output variance vs input variance ────────────────────────────────────
fig_sc, (ax_sc1, ax_sc2) = plt.subplots(1, 2, figsize=(13, 5))
input_vars = input_stds ** 2

for (name, _, _, color) in ACT_FUNCS:
    ax_sc1.plot(input_vars, output_stats[name]["var"],
                color=color, lw=2, marker="o", markersize=5, label=name)
    ax_sc2.plot(input_vars, output_stats[name]["mean"],
                color=color, lw=2, marker="s", markersize=5, label=name)

ax_sc1.plot(input_vars, input_vars, color="k", lw=1.5, ls="--", alpha=0.5,
            label="Identity (Var_out = Var_in)")
ax_sc1.set_xlabel("Input variance $\sigma^2$", fontsize=11)
ax_sc1.set_ylabel("Output variance", fontsize=11)
ax_sc1.set_title("Output Variance vs Input Variance", fontsize=11, fontweight="bold")
ax_sc1.legend(fontsize=8)
ax_sc1.grid(True, alpha=0.25)

ax_sc2.axhline(0, color="k", lw=0.8, ls="--", alpha=0.5)
ax_sc2.set_xlabel("Input variance $\sigma^2$", fontsize=11)
ax_sc2.set_ylabel("Output mean", fontsize=11)
ax_sc2.set_title("Output Mean vs Input Variance (zero-centring check)",
                 fontsize=11, fontweight="bold")
ax_sc2.legend(fontsize=8)
ax_sc2.grid(True, alpha=0.25)

fig_sc.suptitle("Activation Output Statistics vs Input Scale — Xavier/He Init Motivation",
                fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Table: unit Gaussian input statistics ─────────────────────────────────────
unit_x   = rng_sc.normal(0.0, 1.0, 200_000)
print("Output statistics at unit Gaussian input N(0,1):")
print(f"{'Activation':<22s}  {'Mean':>8s}  {'Std':>8s}  {'Var':>8s}  "
      f"{'% > 0':>8s}  {'Zero-centred':>14s}")
print("-" * 78)
for name, fn, _, _ in ACT_FUNCS:
    y_u    = fn(unit_x)
    mean_u = float(y_u.mean())
    std_u  = float(y_u.std())
    var_u  = float(y_u.var())
    pos_u  = float((y_u > 0).mean())
    zc     = "Yes" if abs(mean_u) < 0.05 else "No"
    print(f"{name:<22s}  {mean_u:>8.4f}  {std_u:>8.4f}  {var_u:>8.4f}  "
          f"{pos_u:>8.2%}  {zc:>14s}")

print("\nZero-centred activations (Tanh, LeakyReLU, GELU, Swish) have near-zero mean.")
print("ReLU has ~50% positive outputs and high mean — motivates He initialisation")
print("(He uses gain factor sqrt(2) for ReLU to compensate for the ~50% zeroing).")
print("See 05-08 (Weight Initialisation) for a full treatment.")


---
## Part 4 — Evaluation & Analysis

### 4.1 Activation Distribution Maps

Post-activation value distributions reveal saturation and dead-neuron patterns. We register forward hooks on each hidden layer's activation module, accumulate outputs over several validation batches, and plot histograms.

In [ ]:
# ── Capture post-activation distributions via forward hooks ───────────────────

def collect_activation_distributions(
    act_name:  str,
    n_batches: int = 6,
) -> list[np.ndarray]:
    '''Collect flattened post-activation values from all hidden layers.

    Creates a fresh (randomly initialised) MLP, registers hooks on the
    activation modules, and runs n_batches of validation data through it.

    Args:
        act_name:  Name of the activation function.
        n_batches: Number of validation batches to accumulate.

    Returns:
        List of 1-D NumPy arrays (one per hidden layer) with activation values.
    '''
    act_mod    = get_activation(act_name)
    model_hook = MLP(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES,
                     n_hidden=2, activation=act_mod).to(device)
    captured: list[list[np.ndarray]] = [[], []]
    handles:  list = []

    def _make_hook(layer_idx: int) -> Callable[[nn.Module, tuple, torch.Tensor], None]:
        '''Build a forward hook that appends outputs to captured[layer_idx].

        Args:
            layer_idx: Which slot in captured to fill.

        Returns:
            Hook function compatible with register_forward_hook.
        '''
        def _hook(mod: nn.Module, inp: tuple, out: torch.Tensor) -> None:
            '''Append output tensor to captured buffer.'''
            captured[layer_idx].append(out.detach().cpu().numpy())
        return _hook

    act_types = (nn.ReLU, nn.Sigmoid, nn.Tanh, nn.LeakyReLU, nn.GELU, nn.SiLU)
    act_mods_found = [m for m in model_hook.net if isinstance(m, act_types)]
    for i, lyr in enumerate(act_mods_found[:2]):
        handles.append(lyr.register_forward_hook(_make_hook(i)))

    model_hook.eval()
    with torch.no_grad():
        for batch_idx, (imgs, _) in enumerate(val_loader):
            model_hook(imgs.to(device))
            if batch_idx + 1 >= n_batches:
                break

    for h in handles:
        h.remove()

    return [np.concatenate(c, axis=0).ravel() for c in captured if c]


fig_am, axes_am = plt.subplots(len(ACTIVATION_NAMES), 2, figsize=(12, 17))
for row_idx, (act_name, color) in enumerate(zip(ACTIVATION_NAMES, ACT_COLORS)):
    layer_data = collect_activation_distributions(act_name, n_batches=4)
    for col_idx, vals in enumerate(layer_data):
        ax_am = axes_am[row_idx, col_idx]
        ax_am.hist(vals, bins=60, color=color, alpha=0.75, density=True)
        # annotate dead fraction only for ReLU
        if act_name == "ReLU":
            dead_frac = float((vals == 0.0).mean())
            ax_am.text(0.98, 0.95, f"dead={dead_frac:.1%}",
                       transform=ax_am.transAxes,
                       ha="right", va="top", fontsize=8, color="red")
        # annotate saturation for Sigmoid/Tanh
        if act_name in ("Sigmoid", "Tanh"):
            bound    = 0.9 if act_name == "Sigmoid" else 0.9
            sat_frac = float((np.abs(vals) > bound).mean())
            ax_am.text(0.98, 0.95, f"sat>{bound:.1f}: {sat_frac:.1%}",
                       transform=ax_am.transAxes,
                       ha="right", va="top", fontsize=8, color="red")
        ax_am.set_title(f"{act_name} — Layer {col_idx + 1}", fontsize=9, fontweight="bold")
        ax_am.set_xlabel("Activation value", fontsize=8)
        ax_am.set_ylabel("Density", fontsize=8)
        ax_am.tick_params(labelsize=7)

fig_am.suptitle("Post-Activation Value Distributions (randomly initialised model)",
                fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### 4.2 Per-Layer Gradient Norms During Training

We track the weight gradient norm at each linear layer over the first 25 mini-batches. Healthy gradient flow shows roughly similar norms across layers; vanishing gradients appear as sharply smaller norms in deeper layers.

In [ ]:
# ── Measure gradient norms per linear layer ───────────────────────────────────

def measure_gradient_norms(
    act_name:  str,
    n_batches: int = 25,
) -> dict[str, list[float]]:
    '''Train for n_batches mini-batches and record per-layer weight gradient norms.

    Args:
        act_name:  Activation function name.
        n_batches: Number of mini-batches to process.

    Returns:
        Dict mapping each nn.Linear name to a list of gradient norms (per batch).
    '''
    act_mod  = get_activation(act_name)
    model_gf = MLP(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES,
                   n_hidden=2, activation=act_mod).to(device)
    opt_gf   = torch.optim.Adam(model_gf.parameters(), lr=LEARNING_RATE)
    crit_gf  = nn.CrossEntropyLoss()

    linear_layers: list[tuple[str, nn.Linear]] = [
        (name, mod)
        for name, mod in model_gf.named_modules()
        if isinstance(mod, nn.Linear)
    ]
    norms: dict[str, list[float]] = {name: [] for name, _ in linear_layers}

    model_gf.train()
    for batch_idx, (imgs, labels) in enumerate(train_loader):
        if batch_idx >= n_batches:
            break
        imgs, labels = imgs.to(device), labels.to(device)
        opt_gf.zero_grad()
        loss = crit_gf(model_gf(imgs), labels)
        loss.backward()
        for lname, lmod in linear_layers:
            if lmod.weight.grad is not None:
                norms[lname].append(lmod.weight.grad.norm().item())
        opt_gf.step()

    return norms


gf_data: dict[str, dict[str, list[float]]] = {}
for act_name in ACTIVATION_NAMES:
    gf_data[act_name] = measure_gradient_norms(act_name, n_batches=25)

# ── Bar chart: mean gradient norm per layer per activation ────────────────────
layer_keys     = list(next(iter(gf_data.values())).keys())
n_layers_gf    = len(layer_keys)
bar_width_gf   = 0.13
x_pos_gf       = np.arange(n_layers_gf)
short_names_gf = [lk.split(".")[-1] for lk in layer_keys]

fig_gf, ax_gf = plt.subplots(figsize=(10, 5))
for i, (act_name, color) in enumerate(zip(ACTIVATION_NAMES, ACT_COLORS)):
    means_gf = [float(np.mean(gf_data[act_name][lk])) for lk in layer_keys]
    ax_gf.bar(x_pos_gf + i * bar_width_gf, means_gf,
              width=bar_width_gf, color=color, label=act_name, alpha=0.85)

ax_gf.set_xticks(x_pos_gf + bar_width_gf * (len(ACTIVATION_NAMES) - 1) / 2)
ax_gf.set_xticklabels(short_names_gf, fontsize=9)
ax_gf.set_ylabel("Mean Gradient Norm", fontsize=11)
ax_gf.set_title("Per-Layer Weight Gradient Norms — First 25 Training Batches",
                fontsize=11, fontweight="bold")
ax_gf.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("Mean gradient norms per layer:")
header = f"{'Activation':<12s}" + "".join(f"  {k:<16s}" for k in layer_keys)
print(header)
print("-" * (12 + 18 * n_layers_gf))
for act_name in ACTIVATION_NAMES:
    row = f"{act_name:<12s}"
    for lk in layer_keys:
        row += f"  {float(np.mean(gf_data[act_name][lk])):>16.4f}"
    print(row)

### 4.3 Dead-Neuron & Saturation Analysis

A **dead neuron** is one whose post-activation value is zero for *every* input in the validation set. This affects ReLU neurons with large negative bias. We also measure **saturation fraction** for Sigmoid and Tanh: the proportion of neurons whose activation magnitude exceeds 0.9 (Sigmoid near 1) or 0.9 (Tanh near ±1), where the gradient is near zero.

In [ ]:
# ── Dead & saturated neuron analysis ─────────────────────────────────────────

def analyse_neuron_health(
    act_name:  str,
    threshold: float = 1e-6,
) -> list[dict]:
    '''Measure dead and saturated fractions per hidden layer.

    Args:
        act_name:  Activation function name.
        threshold: Mean |activation| below which a neuron is considered dead.

    Returns:
        List of dicts with keys Activation, Layer, Dead%, Saturated%,
        MeanAbs, StdAbs — one per hidden layer.
    '''
    layer_data = collect_activation_distributions(act_name, n_batches=8)
    rows: list[dict] = []
    for layer_idx, vals_flat in enumerate(layer_data):
        n_neurons    = HIDDEN_DIM
        vals_mat     = vals_flat[:-(len(vals_flat) % n_neurons) or None]
        if len(vals_mat) == 0:
            continue
        n_samples    = len(vals_mat) // n_neurons
        vals_mat     = vals_mat[:n_samples * n_neurons].reshape(n_samples, n_neurons)
        mean_abs_per = np.abs(vals_mat).mean(axis=0)
        dead_frac    = float((mean_abs_per < threshold).mean())
        if act_name == "Sigmoid":
            sat_frac = float((vals_mat > 0.9).mean())
        elif act_name == "Tanh":
            sat_frac = float((np.abs(vals_mat) > 0.9).mean())
        else:
            sat_frac = 0.0
        rows.append({
            "Activation": act_name,
            "Layer":      layer_idx + 1,
            "Dead%":      round(dead_frac * 100, 2),
            "Saturated%": round(sat_frac  * 100, 2),
            "MeanAbs":    round(float(np.abs(vals_mat).mean()), 4),
            "StdAbs":     round(float(np.abs(vals_mat).std()),  4),
        })
    return rows


all_health_rows: list[dict] = []
for act_name in ACTIVATION_NAMES:
    all_health_rows.extend(analyse_neuron_health(act_name))

health_df = pd.DataFrame(all_health_rows)
print("Neuron health analysis (randomly initialised model, validation data):")
print(health_df.to_string(index=False))

# ── Bar chart: dead% and saturated% per activation and layer ──────────────────
fig_nh, axes_nh = plt.subplots(1, 2, figsize=(13, 5))
for layer_idx, ax_nh in enumerate(axes_nh, start=1):
    sub = health_df[health_df["Layer"] == layer_idx]
    x_nh = np.arange(len(sub))
    ax_nh.bar(x_nh - 0.2, sub["Dead%"].values,      width=0.35,
              color=ACT_COLORS, alpha=0.8, label="Dead%")
    ax_nh.bar(x_nh + 0.2, sub["Saturated%"].values, width=0.35,
              color=ACT_COLORS, alpha=0.4, label="Saturated%")
    ax_nh.set_xticks(x_nh)
    ax_nh.set_xticklabels(sub["Activation"].tolist(), fontsize=9, rotation=20)
    ax_nh.set_ylabel("Percentage of neurons (%)", fontsize=10)
    ax_nh.set_title(f"Layer {layer_idx} — Dead & Saturated Neurons",
                    fontsize=11, fontweight="bold")
    ax_nh.legend(fontsize=9)

fig_nh.suptitle("Dead and Saturated Neuron Fractions per Activation (random init)",
                fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nKey: Dead% counts neurons with mean |activation| < 1e-6.")
print("     Saturated% counts Sigmoid > 0.9 or |Tanh| > 0.9 (0 for others).")

# ── Summary: rank activations by combined health score ───────────────────────
print("\nActivation health ranking (freshly initialised, validation data):")
print(f"{'Activation':<12s}  {'Dead L1':>8s}  {'Dead L2':>8s}"
      f"  {'Sat L1':>8s}  {'Sat L2':>8s}  {'Health':>8s}")
print("-" * 62)
for _row in all_health_rows[::2]:   # every other row = layer-1 entries
    _act  = _row["Activation"]
    _l2   = next((r for r in all_health_rows
                  if r["Activation"] == _act and r["Layer"] == 2), {})
    _d1   = _row.get("Dead%", 0.0)
    _d2   = _l2.get("Dead%", 0.0)
    _s1   = _row.get("Saturated%", 0.0)
    _s2   = _l2.get("Saturated%", 0.0)
    # health = 100 minus average dead and saturation percentage
    _h    = 100.0 - 0.5 * (_d1 + _d2) - 0.5 * (_s1 + _s2)
    print(f"{_act:<12s}  {_d1:>8.2f}  {_d2:>8.2f}"
          f"  {_s1:>8.2f}  {_s2:>8.2f}  {_h:>8.2f}")

print("Health = 100 - mean(dead) - mean(saturated).  Higher is better.")
print("Tanh/GELU/Swish score well at random init; ReLU can have high dead%.")
print("After training the picture changes — activations become more uniform.")


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **ReLU dominates practice** due to its simplicity and computational efficiency, but    can produce dead neurons when neurons receive consistently negative inputs; LeakyReLU    and GELU/Swish mitigate this.
2. **Sigmoid and Tanh saturate** for large $|x|$, causing gradients to vanish    exponentially through depth — the simulation shows >$10^{10}\times$ decay at layer 20.
3. **GELU and Swish** are smooth approximations of ReLU that allow small negative outputs    and are the default in modern architectures (GPT, BERT, EfficientNet). They perform    comparably or slightly better than ReLU on FashionMNIST.
4. **Activation choice affects early convergence speed** more than peak accuracy on this    dataset — all non-saturating activations reach similar final test accuracy.
5. **Gradient norm tracking** is a lightweight diagnostic: roughly uniform norms across    layers indicate healthy flow; an order-of-magnitude drop from output to input layer    signals a vanishing gradient issue.

### What's Next

→ **05-04 (Loss Functions)** builds on the training loop here and examines how cross-entropy, MSE, Huber, and focal loss affect learning dynamics.

### Going Further

- Klambauer et al. (2017) — *Self-Normalizing Neural Networks* (SELU preserves unit   normal activations automatically).
- Ramachandran et al. (2017) — *Searching for Activation Functions* (Swish discovered   via neural architecture search).
- Hendrycks & Gimpel (2016) — *Gaussian Error Linear Units (GELUs)* (the GELU paper).